# Vérification des balises

## Chargement des bibliothèques 

In [9]:
import csv
import xml.etree.ElementTree as ET
import os
from collections import defaultdict

## Objectifs de ce Notebook

On veut identifier les erreurs dans chaque fichier XML du corpus
- Vérification des identifiants : 
    * Comparaisons entre les identifiants de l'Index, du sheets et des refs placées dans les textes

Vérification en cours
- vérifier que pers = Personne et place = Lieux
- vérifier que la ref est associé à une des possibilité d'écriture de la liste des variantes ??

## Création des fonctions

### Extractions des persName, variantes et ids

Création des listes et de fichiers csv pour lires les différentes informations recherchées.

On passe tous les fichiers .xml du corpus ArTerm en revue afin d'extraire le texte contenu dans les balises persName pour avoir une liste des variantes utilisées dans le corpus pour chacun des identifiants. Cette liste se trouve dans le fichier `variantes.csv`. 

On obtient également une liste des noms balisés persNames qui n'ont pas été identifiés, ou pas indexés. La liste se trouve dans le fichier `Non_ID.csv`. La sortie est triée par fichier xml.

Enfin on a une liste de toutes les références utilisées dans chaque fichier xml. Cette liste se trouve dans le fichier `refParTexte.csv`.

In [10]:
def load_corpus_files(corpus_file):
    """Charge la liste des fichiers à traiter depuis corpus_peinture.txt"""
    file_list = set()
    try:
        with open(corpus_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:  # Ignore les lignes vides
                    file_list.add(line)
    except FileNotFoundError:
        print(f"Attention: Le fichier {corpus_file} n'a pas été trouvé")
    return file_list

In [11]:
def extract_texts_from_persname(folder_path, sortiePers, NaNPers, refPersInTxt, corpus_files=None):
    variantes_noms = defaultdict(set) # Création de la liste des variantes des noms
    namespace = {'tei': 'http://www.tei-c.org/ns/1.0'} # Namespace des fichiers en XML/TEI pour que le parser fonctionne
    non_identifies = defaultdict(set) # Création de la liste des noms balisés mais non identifiés
    refs_texte = defaultdict(set) # Création d'une liste qui récupère tous les identifiants trouvés dans les balises persName
    
    total_persname = 0
    
    for root_dir, dirs, files in os.walk(folder_path):  # Parcourt récursivement tous les sous-dossiers
        # Exclure les dossiers ITA et FRA
        dirs[:] = [d for d in dirs if d not in ['ITA', 'FRA']]
        
        print("Directory path: %s" % root_dir)  # Correction: root_dir au lieu de root
        print("Directory Names: %s" % dirs)
        print("Files Names: %s" % files)
        for file_name in files:
            if not file_name.startswith('Index') and file_name.endswith('.xml'):
                # Filtre: vérifier si le fichier est dans la liste des fichiers du corpus à traiter
                if corpus_files is not None and file_name not in corpus_files:
                    continue
                    
                file_path = os.path.join(root_dir, file_name)
                print(f"Traitement du fichier : {file_name}")
                try:
                    tree = ET.parse(file_path)
                    root = tree.getroot()
                    
                    persname_count = len(root.findall('.//tei:persName', namespaces=namespace))
                    total_persname += persname_count

                    for name in root.findall('.//tei:persName', namespaces=namespace):  # Trouve toutes les balises persName
                        pers = name.text.strip() if name.text else ''  # Récupère le texte contenu à l'interieur des balises persName
                        ref = name.attrib.get('ref', '').strip()  # Récupère le texte contenu dans l'attribut @ref de chaque persName

                        if not ref:  # Si la balise ne contient pas d'attribut ref on le met dans la liste des non identifiés
                            if pers:  # Seulement si le nom n'est pas vide
                                non_identifies[file_name].add(pers)
                        else:  # Sinon on supprime le # en début d'identifiant et on ajoute l'id et le contenu entre les balises dans la liste des variantes.
                            if ref.startswith('#'):
                                ref = ref[1:]
                            if pers:  # Seulement si le nom n'est pas vide
                                variantes_noms[ref].add(pers)
                            refs_texte[file_name].add(ref)
                    
                except Exception as e:
                    print(f"Erreur dans le fichier {file_name}: {e}")

    sorted_var = sorted(variantes_noms.items(), key=lambda x: len(x[0]), reverse=True) # tri des variantes par ordre alphabétique 
   

    with open(sortiePers, 'w', newline='', encoding='utf-8') as csvfile: # Ecriture d'un fichier csv avec les variantes d'écriture pour chaque indentifiant
        csvwriter = csv.writer(csvfile)
        csvwriter.writerow(['ID', 'Noms'])
        
        for ref, noms in sorted_var:
            
            csvwriter.writerow([ref, ','.join(sorted(noms, key=len, reverse=True))])

    with open(NaNPers, 'w', newline='', encoding='utf-8') as NonDef: # Ecriture d'un fichier csv avec la liste des non identifiés
        inconnu = csv.writer(NonDef)
        inconnu.writerow(['fileName', 'Noms'])

        for file_name, ref in non_identifies.items():
            inconnu.writerow([file_name, ','.join(sorted(ref))])

    with open(refPersInTxt, 'w', newline='', encoding='utf-8') as partxt: # Ecriture d'un fichier csv avec la liste complète des références dans chaque texte.
        persName = csv.writer(partxt)
        persName.writerow(['fileName', 'Ref'])
        
        for file_name, ref in refs_texte.items():
            persName.writerow([file_name, ','.join(sorted(ref))])

    print(f"Nombre total de balises persName trouvées: {total_persname}")

    return variantes_noms, non_identifies, refs_texte

Lien vers les fichiers d'entrée et de sortie de la fonction.

In [12]:
# Le chemin vers les fichiers à traiter avec la fonction. Ici vers le dépôt complet Arterm car les fichiers xml s'y trouve
# Le chemin peut être modifier si la structure du dépôt change.
folder_path = "../corpus-ArTerm"

Liste des variables pour l'extraction des persName

In [13]:
# Noms donnés aux fichiers de sortie. 
# Ils sont dirigés vers un dossier persName.
sortiePers = "persName/variantes.csv"
NaNPers = "persName/Non_ID.csv"
refPersInTxt = 'persName/refParTexte.csv'
erreurPers = "persName/pbParTexte.csv"

Test de la fonction d'extraction, qui fonctionne seule si on veut juste sortir les fichiers de variantes, de pers non identifiés et des refs par texte.

In [14]:
# Charger la liste des fichiers du corpus à traiter depuis corpus_peinture.txt
corpus_file = "corpus_peinture.txt"
corpus_files = load_corpus_files(corpus_file)
print(f"Nombre de fichiers à traiter: {len(corpus_files)}")
print(f"Fichiers: {sorted(corpus_files)}")

Nombre de fichiers à traiter: 19
Fichiers: ['Agucchi_TrattatoPittura.xml', 'Daret_VieRaphael.xml', 'DupuyDuGrez_TraitePeinture.xml', 'Freart_IdeaDellaPerfezione.xml', 'Lomazzo_Idea.xml', 'Lomazzo_TraicteProportion.xml', 'Marino_DicerieSacre.xml', 'Monier_HistoireArtsRapportDessein.xml', 'Pader_LaPeintureParlante.xml', 'Pader_SongeEnigmatique.xml', 'Piles_AbregeViePeintres.xml', 'Piles_ConversationsConnaissancePeinture.xml', 'Piles_CoursPeinture.xml', 'Piles_DialogueColoris.xml', 'Vinci_TraitePeinture_fra.xml', 'Vinci_TrattatoPittura_ITA.xml', 'Zuccari_IdeaPittori.xml', 'Zuccari_Lettera.xml', 'Zuccari_OrigineProgressoAcademiaDissegno.xml']


In [15]:
extract_texts_from_persname(folder_path, sortiePers, NaNPers, refPersInTxt, corpus_files)

Directory path: ../corpus-ArTerm
Directory Names: ['Architecture', 'Peinture', 'Perspective']
Files Names: ['IndexLieux.xml', 'IndexPersonnes.xml', 'tei_arterm.rng']
Directory path: ../corpus-ArTerm\Architecture
Directory Names: []
Files Names: ['Barbaro_DieciLibriDellArchitetturaATIR.xml', 'Bartoli_ArchitetturaATIR.xml', 'Bassi_DispareriArchitettura.xml', 'Cataneo_PrimiQuatroLibriArchitetturaATIR.xml', 'Felibien_PrincipesArchitectureSculpturePeinture.xml', 'Huret_RegleDescrireColomnes.xml', 'Martin_ArchitectureAlberti.xml', 'Martin_ArchitectureSerlio.xml', 'Martin_DiscoursSongePoliphile.xml', 'Palladio_ArchitectureTradFreart.xml', 'Palladio_DescritioneChieseATIR.xml', 'Palladio_QuattroLibriArchitetturaATIR.xml', 'Raphael_LetteraLeoneX.xml', 'Sambin_DiversiteTermesArchitecture.xml', 'Scamozzi_IdeaArchitetturaUniversaleATIR.xml', 'Serlio_LibroExtraodinarioArchitetturaATIR.xml', 'Vignola_RegolaDelliCinqueOrdiniArchitetturaATIR.xml']
Directory path: ../corpus-ArTerm\Peinture
Directory Nam

(defaultdict(set,
             {'SimonGuillainII': {'SIMONE GUILINO PARIGINO',
               'Simone Guilino Francese'},
              'LodovicoGrignani': {'Lodovico Grignani.'},
              'GiovanniAntonioMassani': {'GIOVANNI ATANASIO MOSINI',
               'Giovanni Atanasio Mosini'},
              'AnnibaleCarracci': {'ANNIBAL',
               'ANNIBALE CARRACCI.',
               'Anibal',
               'Annibal',
               'Annibal Carache',
               'Annibal Carracci',
               'Annibal Carrache',
               'Annibale',
               'Annibale Caracci',
               'Annibale Carazzi',
               'Annibale Carracci',
               'Annibale Carracci Bolognesi',
               'Carache',
               'Carracci',
               'Fratello',
               'Hannibal',
               'Hannibal Carache',
               'Maestro',
               "d'Hannibal",
               'il Fratello',
               'le Carache',
               'son frere'},
     

### Extraction des placeName, variantes et ids

Même fonctionnement que la fonction au dessus, appliquée aux balises placeName.
On dirige les fichiers de sorties vers un second dossier nommé `placeName`.

In [16]:
def extract_texts_from_placename(folder_path, sortiePlace, NaNPlace, refPlaceInTxt, corpus_files=None):
    variantes_noms = defaultdict(set)
    namespace = {'tei': 'http://www.tei-c.org/ns/1.0'}
    non_identifies = defaultdict(set)  # should be a set so we can call .add()
    refs_texte = defaultdict(set)
    
    total_placename = 0
    for root_dir, dirs, files in os.walk(folder_path):  # Parcourt récursivement tous les sous-dossiers
        # Exclure les dossiers ITA et FRA
        dirs[:] = [d for d in dirs if d not in ['ITA', 'FRA']]
        
        print("Directory path: %s" % root_dir)  # Correction: root_dir au lieu de root
        print("Directory Names: %s" % dirs)
        print("Files Names: %s" % files)
        for file_name in files:
            if not file_name.startswith('Index') and file_name.endswith('.xml'):
                # Filtre: vérifier si le fichier est dans la liste des fichiers du corpus à traiter
                if corpus_files is not None and file_name not in corpus_files:
                    continue
                    
                file_path = os.path.join(root_dir, file_name)
                print(f"Traitement du fichier : {file_name}")
                try:
                    tree = ET.parse(file_path)
                    root = tree.getroot()
                    
                    placename_count = len(root.findall('.//tei:placeName', namespaces=namespace))
                    total_placename += placename_count

                    for name in root.findall('.//tei:placeName', namespaces=namespace):  # Trouve toutes les balises placeName
                        place = name.text.strip() if name.text else ''  # Récupère le texte contenu à l'interieur des balises placeName
                        ref = name.attrib.get('ref', '').strip()  # Récupère le texte contenu dans l'attribut @ref de chaque placeName

                        if not ref:  # Si la balise ne contient pas d'attribut ref on le met dans la liste des non identifiés
                            if place:  # Seulement si le nom n'est pas vide
                                non_identifies[file_name].add(place)
                        else:  # Sinon on supprime le # en début d'identifiant et on ajoute l'id et le contenu entre les balises dans la liste des variantes.
                            if ref.startswith('#'):
                                ref = ref[1:]
                            if place:  # Seulement si le nom n'est pas vide
                                variantes_noms[ref].add(place)
                            refs_texte[file_name].add(ref)
                    
                except Exception as e:
                    print(f"Erreur dans le fichier {file_name}: {e}")

    sorted_var = sorted(variantes_noms.items(),  key=lambda x: len(x[0]), reverse=True)
    
    with open(sortiePlace, 'w', newline='', encoding='utf-8') as csvfile:
        csvwriter = csv.writer(csvfile)
        csvwriter.writerow(['ID', 'Noms'])
        
        for ref, noms in sorted_var:
            
            csvwriter.writerow([ref, ','.join(sorted(noms, key=len, reverse=True))])

    with open(NaNPlace, 'w', newline='', encoding='utf-8') as NonDef:
        inconnu = csv.writer(NonDef)
        inconnu.writerow(['fileName', 'Noms'])

        for file_name, ref in non_identifies.items():
            inconnu.writerow([file_name, ','.join(sorted(ref))])

    with open(refPlaceInTxt, 'w', newline='', encoding='utf-8') as partxt:
        persName = csv.writer(partxt)
        persName.writerow(['fileName', 'Ref'])
        
        for file_name, ref in refs_texte.items():
            persName.writerow([file_name, ','.join(sorted(ref))])

    print(f"Nombre total de balises placeName trouvées: {total_placename}")

    return variantes_noms, non_identifies, refs_texte

In [17]:
sortiePlace = "placeName/variantes.csv"
NaNPlace = "placeName/Non_ID.csv"
refPlaceInTxt = "placeName/refParTexte.csv"
erreurPlace = "placeName/pbParTexte.csv"

In [18]:
extract_texts_from_placename(folder_path, sortiePlace, NaNPlace, refPlaceInTxt, corpus_files)

Directory path: ../corpus-ArTerm
Directory Names: ['Architecture', 'Peinture', 'Perspective']
Files Names: ['IndexLieux.xml', 'IndexPersonnes.xml', 'tei_arterm.rng']
Directory path: ../corpus-ArTerm\Architecture
Directory Names: []
Files Names: ['Barbaro_DieciLibriDellArchitetturaATIR.xml', 'Bartoli_ArchitetturaATIR.xml', 'Bassi_DispareriArchitettura.xml', 'Cataneo_PrimiQuatroLibriArchitetturaATIR.xml', 'Felibien_PrincipesArchitectureSculpturePeinture.xml', 'Huret_RegleDescrireColomnes.xml', 'Martin_ArchitectureAlberti.xml', 'Martin_ArchitectureSerlio.xml', 'Martin_DiscoursSongePoliphile.xml', 'Palladio_ArchitectureTradFreart.xml', 'Palladio_DescritioneChieseATIR.xml', 'Palladio_QuattroLibriArchitetturaATIR.xml', 'Raphael_LetteraLeoneX.xml', 'Sambin_DiversiteTermesArchitecture.xml', 'Scamozzi_IdeaArchitetturaUniversaleATIR.xml', 'Serlio_LibroExtraodinarioArchitetturaATIR.xml', 'Vignola_RegolaDelliCinqueOrdiniArchitetturaATIR.xml']
Directory path: ../corpus-ArTerm\Peinture
Directory Nam

(defaultdict(set,
             {'Rome': {'ROMA',
               'Roma',
               'Roma.',
               'Romae',
               'Rome',
               'Vile',
               'ancienne Rome',
               'cette Vile',
               'cette Ville',
               'cette belle vile',
               'cette capitale',
               'cette celebre vile',
               'cette illustre vile',
               'cette superbe Ville',
               'cette vile'},
              'Bologne': {'BOLOGNA',
               'Bologna',
               'Bologne',
               'Boulogne',
               'cette vile'},
              'Italie': {'Italia', 'Italie', "l'Italia"},
              'Allemagne': {'Alemagna',
               'Alemagne',
               'Allemagne',
               'Allemange',
               'Germania',
               'Germanie',
               "l'Alemagne"},
              'Flandres': {'Fiandra', 'Flandre', 'Flandres', 'Florentiam'},
              'France': {'France', 'Francia'}

### Création des listes à comparer à partir des fichiers crées depuis l'extraction et des fichiers index (GoogleSheets et XML)

On extrait les informations de `pers.csv` qui contient les identifiants présents sur le GoogleSheets pour les transformer en liste `id_sheets`

On extrait la liste des identifiants récupérés en même temps que les variantes d'écritures des différents noms, depuis le fichier `refParTexte` vers la liste `id_csv`. On utilise ce fichier de sortie afin de conserver l'information des fichiers dans lesquels se trouve chaque ref. 

On extrait les identifiants de l'`IndexPersonnes.xml` dans la liste `id_xml` afin de comparer à la fois les erreurs dans les textes mais également vérifier nos listes d'Index.

In [19]:
def creation_liste_persName(pers_balise_auto, pers_fichiers_xml, pers_index):
    # Création des listes vides
    id_pers_b_auto = []
    id_pers_fichiers_xml = []
    id_pers_index = []
    
    with open(pers_balise_auto, 'r', encoding='utf-8') as sheet: # Lecture du fichier csv qui contient les identifiants du GoogleSheets
        reader = csv.DictReader(sheet)
        for row in reader: 
            if row['ID']:
                id_pers_b_auto.append(row['ID'].strip())
                         
    with open(pers_fichiers_xml, 'r', encoding='utf-8') as cfile: # Lecture du fichier csv qui contient les refs récupérées dans les fichiers xml du corpus
            reader = csv.DictReader(cfile)
            for row in reader: 
                file = row['fileName'].strip() 
                id_noms = row['Ref'].strip()
                if file and id_noms: # on utilise le fichier refParTexte pour garder l'information du classement par fichier pour retrouver les erreurs plus facilement
                    id_noms = [n.strip() for n in id_noms.split(',') if n.strip()]
                    id_pers_fichiers_xml.append({'Nom du fichier': file,'ref': id_noms }) 
    
    tree = ET.parse(pers_index)
    root = tree.getroot()
    namespace = {'tei': 'http://www.tei-c.org/ns/1.0'}
    
    for person in root.findall('.//tei:person', namespaces=namespace): # Lecture du fichier IndexPersonnes pour récupérer les ids présent dans le doc xml
        refindex = person.attrib.get('{http://www.w3.org/XML/1998/namespace}id', '').strip()
        if refindex:
            id_pers_index.append(refindex)


    return id_pers_b_auto, id_pers_fichiers_xml, id_pers_index

Fichiers nécessaires pour la création des listes.  
À ne pas modifier !

In [20]:
pers_balise_auto = "fichiers_noms/auteurs.csv"
pers_fichiers_xml = "persName/refParTexte.csv"
pers_index = "../corpus-ArTerm/IndexPersonnes.xml"

Etape de vérification des listes.

In [21]:
id_pers_b_auto, id_pers_fichiers_xml, id_pers_index = creation_liste_persName(pers_balise_auto, pers_fichiers_xml, pers_index)
print("Liste des IDs de balise auto :", id_pers_b_auto)
print("Liste des IDs de variantes :", id_pers_fichiers_xml)
print("Liste des IDs dans IndexPersonnes:", id_pers_index)

Liste des IDs de balise auto : ['ManiusValeriusMaximusCorvinusMessalla', 'GiovanniBattistaBranconioDellAquila', 'LuciusManliusCapitolinusImperiosus', 'LodovicoDiLeonardoBuonarrotiSimoni', 'ArmandJeanDuPlessisDeRichelieu', 'QuintusFabiusMaximusVerrucosus', 'IsabelleClaireEugenieDAutriche', 'AnthonieVanMontfoortBlocklandt', 'GiovanniPaoloGallucciSalodiano', 'QuintusFabiusMaximusRullianus', 'GastonDeSecondatDeMontesquieu', 'LouisIPhelypeauxDeLaVrilliere', 'GiovanniAndreaGiliodaFabriano', 'PierreLouisReichDePennautier', 'GiovanniBenedettoCastiglione', 'AntonioMariaCiocchiDelMonte', 'GiovanniBattistaDeCavalieri', 'ArmandJeanVignerotDuPlessis', 'LeopoldGuillaumeDeHabsbourg', 'FrancescoBrambillaIlGiovane', 'WolfgangGuillaumeDeNeubourg', 'AlessandroFarneseIlGiovane', 'GiovanniBattistaDellaMarca', 'AntoinePerrenotDeGranvelle', 'CarloLuigiDAragonaTagliava', 'CatherineMichelleDAutriche', 'AntonioDaSangalloIlGiovane', 'AntonioDaSangalloIlVecchio', 'GiovanniFrancescoRomanelli', 'MarcDeVulsonDeLaCol

On extrait les informations depuis le fichier `lieux.csv`

On créer des listes pour comparer les refs et ids des placeName de la même façon que pour les persName.

In [22]:
def creation_liste_placeName(place_balise_auto, place_fichiers_xml, place_index):
    id_place_b_auto = []
    id_place_fichiers_xml = []
    id_place_index = []
    
    with open(place_balise_auto, 'r', encoding='utf-8') as sheet:
        reader = csv.DictReader(sheet)
        for row in reader: 
            if row['ID']:
                id_place_b_auto.append(row['ID'].strip())
                         
    with open(place_fichiers_xml, 'r', encoding='utf-8') as cfile:
            reader = csv.DictReader(cfile)
            for row in reader: 
                file = row['fileName'].strip() 
                id_noms = row['Ref'].strip()
                if file and id_noms: 
                    id_noms = [n.strip() for n in id_noms.split(',') if n.strip()]
                    id_place_fichiers_xml.append({'Nom du fichier': file,'ref': id_noms }) 
    
    tree = ET.parse(place_index)
    root = tree.getroot()
    namespace = {'tei': 'http://www.tei-c.org/ns/1.0'}
    
    for person in root.findall('.//tei:place', namespaces=namespace):
        refindex = person.attrib.get('{http://www.w3.org/XML/1998/namespace}id', '').strip()
        if refindex:
            id_place_index.append(refindex)


    return id_place_b_auto, id_place_fichiers_xml, id_place_index

In [23]:
place_balise_auto = "fichiers_noms/lieux.csv"
place_fichiers_xml = "placeName/refParTexte.csv"
place_index = "../corpus-ArTerm/IndexLieux.xml"

In [24]:
id_place_b_auto, id_place_fichiers_xml, id_place_index = creation_liste_placeName(place_balise_auto, place_fichiers_xml, place_index)
print("Liste des IDs de balise auto :", id_place_b_auto)
print("Liste des IDs des textes XML :", id_place_fichiers_xml)
print("Liste des IDs dans IndexLieux:", id_place_index)

Liste des IDs de balise auto : ['SaintRemyDeProvence', 'SaintEtienneVille', 'SaintQuentinVille', 'BassanoDelGrappa', 'SantAngeloInVado', 'CittaDiCastello', 'Constantinople', 'PordenoneVille', 'ValleDiBlenio', 'Fontainebleau', 'Grottaferrata', 'AixLaChapelle', 'NeubourgDuche', 'GambassiTerme', 'MonteCavallo', 'EmpireOrient', 'ThebesEgypte', 'CastelFranco', 'FrancheComte', 'Ripatransone', 'PortoErcole', 'Schaffhouse', 'Montpellier', 'Portovenere', 'Versailles', 'Mauritanie', 'Angleterre', 'Strasbourg', 'Pessinonte', 'Alexandrie', 'Samothrace', 'Beauregard', 'Caravaggio', 'Settignano', 'Flessingue', 'Westphalie', 'Ingolstadt', 'Wurtzbourg', 'Heidelberg', 'Copenhague', 'Oudenaarde', 'Monferrato', 'Allemagne', 'Lombardie', 'Plaisance', 'Nuremberg', 'Languedoc', 'Marseille', 'Albigeois', 'Valduggia', 'Jerusalem', 'Metaponte', 'MontLiban', 'Pouzzoles', 'Augsbourg', 'Bruxelles', 'Amsterdam', 'Macedoine', 'Belvedere', 'Thessalie', 'Caprarola', 'Correggio', 'Heemskerk', 'Meulebeke', 'Francfort',

#### Comparaisons entre les trois listes

On peut visualiser toutes les différences entre les listes. On ne sort en fichier externe que les identifiants trouvés dans les textes mais qui ne sont pas présent dans le XML. Ce fichier est `pbParTexte.csv` qui permet de voir les erreurs d'identifiants par texte. 

Ces erreurs peuvent être de plusieurs nature :
* L'identifiant n'existe pas dans le XML, oubli ou suppression dans le XML
* L'identifiant est erroné 
* L'identifiant appartient à un lieu et à été mal balisé

In [25]:
def comparaison(pers_balise_auto, pers_fichiers_xml, pers_index, erreurPers, place_balise_auto, place_fichiers_xml, place_index, erreurPlace):

    id_pers_b_auto, id_pers_fichiers_xml, id_pers_index = creation_liste_persName(pers_balise_auto, pers_fichiers_xml, pers_index)
    id_place_b_auto, id_place_fichiers_xml, id_place_index = creation_liste_placeName(place_balise_auto, place_fichiers_xml, place_index)
    
    id_pers_refs_fichiers_xml = [] # Dans les listes id_csv on a conservé un dictionnaire pour trier les refs par texte. On récupére en une seule liste toutes les références pour les comparer et pouvoir reclasser les refs par textes plus tard
    for entry in id_pers_fichiers_xml:
        id_pers_refs_fichiers_xml.extend(entry['ref'])
    
    id_place_refs_fichiers_xml = []
    for entry in id_place_fichiers_xml:
        id_place_refs_fichiers_xml.extend(entry['ref'])

   # On peut étudier toutes les différences entre toutes les listes que l'on a.

    sheetVStxt_pers = list(set(id_pers_b_auto) - set(id_pers_refs_fichiers_xml))
    sheetVStxt_place = list(set(id_place_b_auto) - set(id_place_refs_fichiers_xml))
    #print("\nIdentifiants du sheet qui ne sont pas présents dans les textes:\n", sheetVStxt_pers, "\n", sheetVStxt_place)

    sheetVSpers_index = list(set(id_pers_b_auto) - set(id_pers_index))
    sheetVSplace_index = list(set(id_place_b_auto) - set(id_place_index))
    #print("\nIdentifiants du sheet qui ne sont pas présents dans l'index':\n", sheetVSpers_index, "\n", sheetVSplace_index)

    txtVSpers_index = list(set(id_pers_refs_fichiers_xml) - set(id_pers_index))
    txtVSplace_index = list(set(id_place_refs_fichiers_xml) - set(id_place_index))
    print("\nIdentifiants des textes qui ne sont pas présents dans l'index':\n", txtVSpers_index, "\n", txtVSplace_index)

    xmlVStxt_pers = list(set(id_pers_index) - set(id_pers_refs_fichiers_xml))
    xmlVStxt_place = list(set(id_place_index) - set(id_place_refs_fichiers_xml))
    print("\nIdentifiants de l'index qui ne sont pas présents dans les textes':\n", xmlVStxt_pers, "\n", xmlVStxt_place)

    txtVSpers_balise_auto = list(set(id_pers_refs_fichiers_xml) - set(id_pers_b_auto))
    txtVSplace_balise_auto = list(set(id_place_refs_fichiers_xml) - set(id_place_b_auto))
    #print("\nIdentifiants des textes qui ne sont pas présents dans le sheet:\n", txtVSpers_balise_auto, "\n", txtVSplace_balise_auto)

    xmlVSpers_balise_auto = list(set(id_pers_index) - set(id_pers_b_auto))
    xmlVSplace_balise_auto = list(set(id_place_index) - set(id_place_b_auto))
    #print("\nIdentifiants de l'index qui ne sont pas présents dans le sheet:\n", xmlVSpers_balise_auto, "\n", xmlVSplace_balise_auto)

    # On écrit deux fichiers csv avec les identifiants qui sont dans les textes mais pas dans l'IndexPersonnes ou Lieux.
    with open(erreurPers, 'w', newline='', encoding='utf-8') as ErreurPers:
            pb = csv.writer(ErreurPers)
            pb.writerow(['fileName', 'Ref'])
            
            for entry in id_pers_fichiers_xml:
                file_name = entry['Nom du fichier']
                refs_diff = [ref for ref in entry['ref'] if ref in txtVSpers_index]
                if refs_diff:
                    pb.writerow([file_name, ','.join(sorted(refs_diff))])

    with open(erreurPlace, 'w', newline='', encoding='utf-8') as ErreurPlace:
            pb = csv.writer(ErreurPlace)
            pb.writerow(['fileName', 'Ref'])
            
            for entry in id_place_fichiers_xml:
                file_name = entry['Nom du fichier']
                refs_diff = [ref for ref in entry['ref'] if ref in txtVSplace_index]
                if refs_diff:
                    pb.writerow([file_name, ','.join(sorted(refs_diff))])
                
    return sheetVStxt_pers, sheetVSpers_index, txtVSpers_index, txtVSpers_balise_auto, xmlVSpers_balise_auto, xmlVStxt_pers, sheetVStxt_place, sheetVSplace_index, txtVSplace_index, txtVSplace_balise_auto, xmlVSplace_balise_auto, xmlVStxt_place

### Executer le programme.

Toutes les variables attendues par la fonction `comparaison()` ont été définies plus tôt pour l'utilisation des autres fonctions. Il suffit donc d'exécuter cette dernière cellule pour lancer la comparaison des différentes listes.

In [26]:
comparaison(pers_balise_auto, pers_fichiers_xml, pers_index, erreurPers, place_balise_auto, place_fichiers_xml, place_index, erreurPlace)


Identifiants des textes qui ne sont pas présents dans l'index':
 ['GottardoDaPonte', 'OrazioGentileschi', 'DonatoZeno', 'RaffaeleSansoniRiario', 'GiovanniAntonioSangiorgio', 'FedeGalitia', 'Ambroise', 'MilondeCrotone', 'FranciaBigio', 'DenysdeColophon', 'GiovanniBattistaFonteo', 'SainteAgathe', 'Caporali', 'PaoloSaetoniGrimaldi', 'Eutychides', 'OratioSammacchini', 'NuntioGalitia', 'AgostinoTassi', 'FedericoComandino', 'CaterinaPenni', 'Allemagne', 'Sigonius', 'Gaudentio', 'GirolamodaTreviso', 'RuggieroDeRuggieri', 'MichelAngeJugementSixtine', 'Pireico', 'DonRodrigoDiToledo', 'LodovicoDomenichi', 'VincenzoBandello', 'SigismondoFoliano', 'Colote2', 'GiovanniAlberti2', 'MarfisaEste', 'Heraclite', 'SainteApolline', 'Jeremie', 'GuidoReni', 'Muses'] 
 ['Cumes', 'Suisse', 'Turin', 'Spolete', 'Istrie', 'Loire', 'ForumBoarium', 'Kos', 'Bruegel', 'Brie', 'Chiusi', 'Cana', 'AbbayeSaintMartinEsAiresDeTroyes', 'Euphranor', 'TriniteDuMont', 'Arabie', 'PantheonRome', 'Lydie', 'MonteCaucaso', 'PlaceD

(['Denys',
  'Iolaos',
  'PseudoNechepsos',
  'Apama',
  'LaodiceDeMacedoine',
  'Tyrannoctones',
  'HermolaoBarbaro',
  'Phyromachos',
  'LeonardoGarzoni',
  'Epaminondas',
  'Pymandre',
  'Anubis',
  'TitusTatius',
  'GianVittorioRossi',
  'TheodoreDeSamos',
  'NicoloMartinelli',
  'ArtemisiaGentileschi',
  'AulusCorneliusCelsus',
  'Deioces',
  'JeanMace',
  'Phraortes',
  'Rhadamanthe',
  'Othon',
  'ElisabethDAragon',
  'CaiusGracchus',
  'GiovanniGirolamoMorone',
  'DariusI',
  'SainteConstance',
  'Parmenion',
  'Themistocle',
  'Kheops',
  'Censorin',
  'LouisIDuGuernier',
  'Epimenide',
  'BlaiseFrancoisPagan',
  'Barnaba',
  'FlaviusJosephe',
  'ApollineDAlexandrie',
  'Picus',
  'HydreDeLerne',
  'SantAmbroise',
  'PubliusCorneliusDolabella',
  'AntoineDeVille',
  'GeorgiusAgricola',
  'Euryalus',
  'AntoineLePautre',
  'LuciusAureliusVerus',
  'Tenes',
  'AppiusClaudiusCaecus',
  'JeanJulesArmandColbert',
  'PhilippeIIIDEspagne',
  'Paciotto',
  'DenysHalicarnasse',
  'Paol

## Comparaison entre les index et les refs utilisées

On sépare les différentes erreurs :
* les personnes annotées avec des balises placeName
* les lieux annotés avec des balises persName
* les identifiants qui ne figurent dans aucuns Index.

In [27]:
def cross_comparaison(pers_balise_auto, pers_fichiers_xml, pers_index, place_balise_auto, place_fichiers_xml, place_index, erreurs):

    id_pers_b_auto, id_pers_fichiers_xml, id_pers_index = creation_liste_persName(pers_balise_auto, pers_fichiers_xml, pers_index)
    id_place_b_auto, id_place_fichiers_xml, id_place_index = creation_liste_placeName(place_balise_auto, place_fichiers_xml, place_index)
    
    id_pers_refs_fichiers_xml = []
    for entry in id_pers_fichiers_xml:
        id_pers_refs_fichiers_xml.extend(entry['ref'])
    
    id_place_refs_fichiers_xml = []
    for entry in id_place_fichiers_xml:
        id_place_refs_fichiers_xml.extend(entry['ref'])

    # Liste des identifiants des listes tirées des texte qui ne sont pas dans les indexs
    ID_inexistant = [x for x in id_pers_refs_fichiers_xml + id_place_refs_fichiers_xml if x not in id_pers_index and x not in id_place_index]
    # Liste des identifiants dans des balises persName qui sont dans l'IndexLieux (balisé avec persName au lieu de placeName)
    place_dans_PersName = [x for x in id_pers_refs_fichiers_xml if x not in id_pers_index and x in id_place_index]
    # Liste des identifiants dans des balises placeName qui sont dans l'IndexPersonnes (balisé avec placeName au lieu de persName)
    pers_dans_PlaceName = [x for x in id_place_refs_fichiers_xml if x not in id_place_index and x in id_pers_index]
    
    # On écrit un fichier csv avec les trois listes créées au dessus pour afficher les résultats plus clairement avec un tri par fichier
    with open(erreurs, 'w', newline='', encoding='utf-8') as Erreur:
            pb = csv.writer(Erreur)
            pb.writerow(["Identifiants inexistants\n"])
            pb.writerow(['fileName', 'Ref'])
            
            for entry in id_place_fichiers_xml + id_pers_fichiers_xml:
                file_name = entry['Nom du fichier']
                refs_diff = [ref for ref in entry['ref'] if ref in ID_inexistant]
                if refs_diff:
                    pb.writerow([file_name, ','.join(sorted(refs_diff))])
            
            pb.writerow(["\nLieux dans des balises persName\n"])
            for entry in id_pers_fichiers_xml:
                file_name = entry['Nom du fichier']
                refs_diff = [ref for ref in entry['ref'] if ref in place_dans_PersName]
                if refs_diff:
                    pb.writerow([file_name, ','.join(sorted(refs_diff))])

            pb.writerow(["\nPersonnes dans des balises placeName\n"])
            for entry in id_place_fichiers_xml:
                file_name = entry['Nom du fichier']
                refs_diff = [ref for ref in entry['ref'] if ref in pers_dans_PlaceName]
                if refs_diff:
                    pb.writerow([file_name, ','.join(sorted(refs_diff))])

    print("\nIdentifiants trouvé dans des balises persName qui sont dans IndexLieux: \n", place_dans_PersName)
    print("\nIdentifiants trouvé dans des balises place qui sont dans IndexPersonnes: \n", pers_dans_PlaceName)
    print("\nLes identifiants n'existent dans aucun index XML:\n", ID_inexistant)


    return ID_inexistant, place_dans_PersName, pers_dans_PlaceName

Un fichier de sortie unique qui sépare les différentes erreurs tout en indiquant dans quel fichier XML elles se trouvent.

In [28]:
erreurs = "erreurs.csv"

## Utilisation de la fonction pour cross check les listes 

In [29]:
cross_comparaison(pers_balise_auto, pers_fichiers_xml, pers_index, place_balise_auto, place_fichiers_xml, place_index, erreurs)


Identifiants trouvé dans des balises persName qui sont dans IndexLieux: 
 ['Allemagne']

Identifiants trouvé dans des balises place qui sont dans IndexPersonnes: 
 ['Euphranor']

Les identifiants n'existent dans aucun index XML:
 ['GiovanniAntonioSangiorgio', 'Muses', 'GuidoReni', 'GuidoReni', 'DonRodrigoDiToledo', 'FedeGalitia', 'FedericoComandino', 'Gaudentio', 'GiovanniBattistaFonteo', 'GottardoDaPonte', 'NuntioGalitia', 'OratioSammacchini', 'SigismondoFoliano', 'GuidoReni', 'Ambroise', 'Eutychides', 'Heraclite', 'Jeremie', 'Pireico', 'Caporali', 'GuidoReni', 'GuidoReni', 'SainteAgathe', 'SainteApolline', 'GuidoReni', 'OrazioGentileschi', 'AgostinoTassi', 'CaterinaPenni', 'FranciaBigio', 'GirolamodaTreviso', 'GuidoReni', 'MichelAngeJugementSixtine', 'OrazioGentileschi', 'RaffaeleSansoniRiario', 'RuggieroDeRuggieri', 'VincenzoBandello', 'GuidoReni', 'DonatoZeno', 'GuidoReni', 'Colote2', 'DenysdeColophon', 'GiovanniAlberti2', 'LodovicoDomenichi', 'MilondeCrotone', 'Sigonius', 'Marfis

(['GiovanniAntonioSangiorgio',
  'Muses',
  'GuidoReni',
  'GuidoReni',
  'DonRodrigoDiToledo',
  'FedeGalitia',
  'FedericoComandino',
  'Gaudentio',
  'GiovanniBattistaFonteo',
  'GottardoDaPonte',
  'NuntioGalitia',
  'OratioSammacchini',
  'SigismondoFoliano',
  'GuidoReni',
  'Ambroise',
  'Eutychides',
  'Heraclite',
  'Jeremie',
  'Pireico',
  'Caporali',
  'GuidoReni',
  'GuidoReni',
  'SainteAgathe',
  'SainteApolline',
  'GuidoReni',
  'OrazioGentileschi',
  'AgostinoTassi',
  'CaterinaPenni',
  'FranciaBigio',
  'GirolamodaTreviso',
  'GuidoReni',
  'MichelAngeJugementSixtine',
  'OrazioGentileschi',
  'RaffaeleSansoniRiario',
  'RuggieroDeRuggieri',
  'VincenzoBandello',
  'GuidoReni',
  'DonatoZeno',
  'GuidoReni',
  'Colote2',
  'DenysdeColophon',
  'GiovanniAlberti2',
  'LodovicoDomenichi',
  'MilondeCrotone',
  'Sigonius',
  'MarfisaEste',
  'PaoloSaetoniGrimaldi',
  'Loire',
  'Cana',
  'Lydie',
  'Turin',
  'Arabie',
  'Cumes',
  'Kos',
  'Acre',
  'Tarn',
  'AbbayeSa

# Comparaison variantes

In [30]:
var_textes_pers = "persName/variantes.csv"
var_textes_place = "placeName/variantes.csv"
var_index_pers = "../corpus-ArTerm/IndexPersonnes.xml"
var_index_place = "../corpus-ArTerm/IndexLieux.xml"

with open(var_textes_pers, 'r', encoding='utf-8') as var_pers, open(var_textes_place, 'r', encoding='utf-8') as var_place:
    reader_pers = csv.DictReader(var_pers)
    reader_place = csv.DictReader(var_place)
    
    var_pers_dict = {row['ID'].strip(): [n.strip() for n in row['Noms'].split(',') if n.strip()] for row in reader_pers}
    var_place_dict = {row['ID'].strip(): [n.strip() for n in row['Noms'].split(',') if n.strip()] for row in reader_place}

tree_pers = ET.parse(var_index_pers)
root_pers = tree_pers.getroot()
namespace = {'tei': 'http://www.tei-c.org/ns/1.0'}
index_pers_dict = {}
for person in root_pers.findall('.//tei:person', namespaces=namespace):
    refindex = person.attrib.get('{http://www.w3.org/XML/1998/namespace}id', '').strip()
    if refindex:
        index_pers_dict[refindex] = [name.text.strip() for name in person.findall('.//tei:addName', namespaces=namespace) if name.text and name.text.strip()]
    else:
        index_pers_dict[refindex] = [name.text.strip() for name in person.findall('.//tei:forename', namespaces=namespace) if name.text and name.text.strip()]   
tree_place = ET.parse(var_index_place)
root_place = tree_place.getroot()   
index_place_dict = {}
for place in root_place.findall('.//tei:place', namespaces=namespace):
    refindex = place.attrib.get('{http://www.w3.org/XML/1998/namespace}id', '').strip()
    if refindex:
        index_place_dict[refindex] = [name.text.strip() for name in place.findall('.//tei:placeName', namespaces=namespace) if name.text and name.text.strip()]

# Afficher les listes créees pour vérifier les résultats
print("\nVariantes d'écriture pour les personnes:\n", var_pers_dict)
print("\nVariantes d'écriture pour les lieux:\n", var_place_dict)
print("\nVariantes d'écriture dans l'index pour les personnes:\n", index_pers_dict)
print("\nVariantes d'écriture dans l'index pour les lieux:\n", index_place_dict)



Variantes d'écriture pour les personnes:
 {'ManiusValeriusMaximusCorvinusMessalla': ['Valerius Messala', 'Mexala'], 'GiovanniBattistaBranconioDellAquila': ["Jean Baptiste de l'Aquila"], 'LodovicoDiLeonardoBuonarrotiSimoni': ['Loüis Bonarotti Simoni'], 'LuciusManliusCapitolinusImperiosus': ['Lucio Manilio'], 'QuintusFabiusMaximusVerrucosus': ['Fabius Maximus', 'F.Maximus', 'Fabius'], 'ArmandJeanDuPlessisDeRichelieu': ['Cardinal Duc de Richelieu', 'Cardinale di Richelieu', 'Cardinal de Richelieu', 'M. le Cardinal'], 'IsabelleClaireEugenieDAutriche': ["l'lnfante Isabelle", 'Infante Isabelle', 'Princesse', "l'Infante", 'Infante'], 'AnthonieVanMontfoortBlocklandt': ['Antoine de Montfort de Blocland'], 'GiovanniPaoloGallucciSalodiano': ['M. Giovanni Paolo Gallucci Salodiano'], 'QuintusFabiusMaximusRullianus': ['Quintus Fabius Rutilien'], 'GastonDeSecondatDeMontesquieu': ['Baron de Montesquieu'], 'LouisIPhelypeauxDeLaVrilliere': ['Monsieur de'], 'GiovanniAndreaGiliodaFabriano': ['Messer Giov

In [31]:
# Comparaison des variantes d'écriture trouvées dans les textes avec celles présentes dans les index pour vérifier si les variantes trouvées dans les textes sont déjà présentes dans les index ou si elles sont nouvelles et peuvent être ajoutées à l'index.

for ref, noms in var_pers_dict.items():
    if ref in index_pers_dict:
        index_noms = set(index_pers_dict[ref])
        new_noms = [n for n in noms if n not in index_noms]
        if new_noms:
            print(f"\nPour l'identifiant {ref} des personnes, les variantes d'écriture suivantes sont nouvelles et peuvent être ajoutées à l'index:\n", new_noms)
    else:
        print(f"\nL'identifiant {ref} des personnes n'est pas présent dans l'index, les variantes d'écriture suivantes peuvent être ajoutées à l'index:\n", noms)
        



Pour l'identifiant GiovanniBattistaBranconioDellAquila des personnes, les variantes d'écriture suivantes sont nouvelles et peuvent être ajoutées à l'index:
 ["Jean Baptiste de l'Aquila"]

Pour l'identifiant ArmandJeanDuPlessisDeRichelieu des personnes, les variantes d'écriture suivantes sont nouvelles et peuvent être ajoutées à l'index:
 ['M. le Cardinal']

Pour l'identifiant IsabelleClaireEugenieDAutriche des personnes, les variantes d'écriture suivantes sont nouvelles et peuvent être ajoutées à l'index:
 ['Princesse', "l'Infante", 'Infante']

Pour l'identifiant LouisIPhelypeauxDeLaVrilliere des personnes, les variantes d'écriture suivantes sont nouvelles et peuvent être ajoutées à l'index:
 ['Monsieur de']

Pour l'identifiant PierreLouisReichDePennautier des personnes, les variantes d'écriture suivantes sont nouvelles et peuvent être ajoutées à l'index:
 ['Monsieur de']

Pour l'identifiant FrancescoBrambillaIlGiovane des personnes, les variantes d'écriture suivantes sont nouvelles e

In [32]:
for ref, noms in var_place_dict.items():
    if ref in index_place_dict:
        index_noms = set(index_place_dict[ref])
        new_noms = [n for n in noms if n not in index_noms]
        if new_noms:
            print(f"\nPour l'identifiant {ref} des lieux, les variantes d'écriture suivantes sont nouvelles et peuvent être ajoutées à l'index:\n", new_noms)
    else:
        print(f"\nL'identifiant {ref} des lieux n'est pas présent dans l'index, les variantes d'écriture suivantes peuvent être ajoutées à l'index:\n", noms)   




L'identifiant AbbayeSaintMartinEsAiresDeTroyes des lieux n'est pas présent dans l'index, les variantes d'écriture suivantes peuvent être ajoutées à l'index:
 ['Abbaye de saint Martin de Troyes']

L'identifiant BasiliqueSantaCroceFlorence des lieux n'est pas présent dans l'index, les variantes d'écriture suivantes peuvent être ajoutées à l'index:
 ['Eglise de Sainte Croix']

Pour l'identifiant SaintRemyDeProvence des lieux, les variantes d'écriture suivantes sont nouvelles et peuvent être ajoutées à l'index:
 ['Saint Remi']

Pour l'identifiant SaintEtienneVille des lieux, les variantes d'écriture suivantes sont nouvelles et peuvent être ajoutées à l'index:
 ['Saint Etienne']

Pour l'identifiant SaintQuentinVille des lieux, les variantes d'écriture suivantes sont nouvelles et peuvent être ajoutées à l'index:
 ['Saint Quentin']

Pour l'identifiant BassanoDelGrappa des lieux, les variantes d'écriture suivantes sont nouvelles et peuvent être ajoutées à l'index:
 ['Bassan']

Pour l'identifi

## Variantes identiques pour plusieurs identifiants différents 

In [33]:
# Voir si variantes identiques pour plusieurs identifiants différents pour vérifier les doublons d'identifiants dans les index ou les erreurs de balisage dans les textes.
inverse_var_pers = defaultdict(set)
for ref, noms in var_pers_dict.items():
    for nom in noms:
        inverse_var_pers[nom].add(ref)
inverse_var_place = defaultdict(set)
for ref, noms in var_place_dict.items():
    for nom in noms:
        inverse_var_place[nom].add(ref)
print("\nVariantes d'écriture identiques pour plusieurs identifiants différents pour les personnes:\n", {nom: refs for nom, refs in inverse_var_pers.items() if len(refs) > 1})
print("\nVariantes d'écriture identiques pour plusieurs identifiants différents pour les lieux:\n", {nom: refs for nom, refs in inverse_var_place.items() if len(refs) > 1})

# faire une sortie des résultats plus claire pour faciliter la lecture
print("\nVariantes d'écriture identiques pour plusieurs identifiants différents pour les personnes:")
for nom, refs in inverse_var_pers.items():
    if len(refs) > 1:
        print(f"  {nom}: {refs}")
print("\nVariantes d'écriture identiques pour plusieurs identifiants différents pour les lieux:")
for nom, refs in inverse_var_place.items():
    if len(refs) > 1:
        print(f"  {nom}: {refs}")



Variantes d'écriture identiques pour plusieurs identifiants différents pour les personnes:
 {'Fabius': {'GaiusFabiusPictor', 'QuintusFabiusMaximusVerrucosus'}, 'Princesse': {'IsabelleClaireEugenieDAutriche', 'Didon', 'Semiramis'}, 'Monsieur de': {'LouisHesselin', 'PierreLouisReichDePennautier', 'LouisIPhelypeauxDeLaVrilliere', 'AntoineDeRatabon', 'MichelDeMarolles'}, 'François': {'FransPourbusI', 'FrancoisIer', 'FrancescoBassano', 'FrancescoTorbido', 'FrancescoSalviati', 'FransFlorisII', 'FrancescoBrambillaIlGiovane', 'FrancescoFrancia'}, 'Cardinal Farnése': {'AlessandroFarneseIlGiovane', 'EdouardFarnese'}, 'Antoine': {'AntonioDaSangalloIlVecchio', 'AntonelloDeMessine', 'AntonioDaSangalloIlGiovane', 'AntonioCarracci', 'AntonioPollaiuolo', 'AntonioCampi'}, 'Giacomo': {'JacopoBassano', 'GiovanniGiacomoDellaPorta'}, 'Michelange': {'MichelAngeJugementSixtine', 'MichelAnge'}, 'Cardinal': {'CharlesDeLorraine', 'BernardoDoviziDaBibbiena', 'HippolyteIIEste'}, 'Jean Baptiste': {'GiovanniBatti

In [34]:
# Comparer les variantes de lieux et de personnes pour identifier les potentielles ambiguités entre les deux index (ex: même variante d'écriture pour un lieu et une personne différente) pour vérifier les erreurs de balisage dans les textes ou les doublons d'identifiants dans les index.
ambiguites = defaultdict(set)
for nom, refs in inverse_var_pers.items():
    if nom in inverse_var_place:
        ambiguites[nom].update(refs)
        ambiguites[nom].update(inverse_var_place[nom])
print("\nVariantes d'écriture identiques pour des identifiants de personnes et de lieux différents:\n", {nom: refs for nom, refs in ambiguites.items() if len(refs) > 1})

print("\nVariantes d'écriture identiques pour des identifiants de personnes et de lieux différents:")
for nom, refs in ambiguites.items():
    if len(refs) > 1:
        print(f"  {nom}: {refs}")




Variantes d'écriture identiques pour des identifiants de personnes et de lieux différents:
 {'Troye': {'AntoineNicolasDeTroy', 'Troie'}, 'Hemskerc': {'MaertenVanHeemskerck', 'Heemskerk'}, 'Champagne': {'Champagne', 'PhilippeDeChampaigne'}, 'Baviere': {'Baviere', 'BavieroDeCarocci'}, 'Francia': {'France', 'FrancescoFrancia'}, 'Europe': {'Europe', 'PrincesseEurope', 'EuropaAnguissola'}, 'Vinci': {'Vinci', 'LeonardoDaVinci'}, 'PARIS': {'ParisMythologie', 'Paris'}, 'Paris': {'ParisMythologie', 'Paris'}, 'Israel': {'IsraelSilvestre', 'Israel'}, 'Luca': {'Lucques', 'LucaSignorelli'}, 'Bassan': {'JacopoBassano', 'BassanoDelGrappa', 'Bassano'}, 'Saint Etienne': {'SaintEtienneVille', 'SaintEtienne'}, 'Cambassi': {'LucaCambiaso', 'GambassiTerme'}, 'Schorel': {'JanVanScorel', 'Schoorl'}, 'Loire': {'Loire', 'NicolasLoir'}, 'Mabuse': {'Maubeuge', 'JanGossaert'}, 'Brugle': {'Bruegel', 'JanBrueghel'}, 'Pordenone': {'PordenoneVille', 'Pordenone'}, 'Pordenon': {'PordenoneVille', 'Pordenone'}, 'Caravag

In [35]:
# comparaison avec auteurs.csv et lieux.csv pour vérifier si les variantes trouvés dans les textes sont dans les listes à baliser pour voir lesquels peuvent manquer pour le balisage automatique.
with open(pers_balise_auto, 'r', encoding='utf-8') as sheet:
    reader = csv.DictReader(sheet)
    pers_balise_auto_dict = {row['ID'].strip(): row['Noms'].strip() for row in reader if row['ID'] and row['Noms']}
with open(place_balise_auto, 'r', encoding='utf-8') as sheet:
    reader = csv.DictReader(sheet)
    place_balise_auto_dict = {row['ID'].strip(): row['Noms'].strip() for row in reader if row['ID'] and row['Noms']}
    for ref, noms in var_pers_dict.items():
        if ref in pers_balise_auto_dict:
            sheet_noms = set(pers_balise_auto_dict[ref].split(','))
            new_noms = [n for n in noms if n not in sheet_noms]
            if new_noms:
                print(f"\nPour l'identifiant {ref} des personnes, les variantes d'écriture suivantes ne sont pas dans la liste de balisage auto et peuvent être ajoutées pour le balisage automatique:\n", new_noms)
        else:
            print(f"\nL'identifiant {ref} des personnes n'est pas présent dans le balisage auto, les variantes d'écriture suivantes peuvent être ajoutées pour le balisage automatique:\n", noms)
    for ref, noms in var_place_dict.items():
        if ref in place_balise_auto_dict:
            sheet_noms = set(place_balise_auto_dict[ref].split(','))
            new_noms = [n for n in noms if n not in sheet_noms]
            if new_noms:
                print(f"\nPour l'identifiant {ref} des lieux, les variantes d'écriture suivantes ne sont pas dans la liste de balisage auto et peuvent être ajoutées pour le balisage automatique:\n", new_noms)
        else:
            print(f"\nL'identifiant {ref} des lieux n'est pas présent dans le balisage auto, les variantes d'écriture suivantes peuvent être ajoutées pour le balisage automatique:\n", noms)

            



Pour l'identifiant GiovanniBattistaBranconioDellAquila des personnes, les variantes d'écriture suivantes ne sont pas dans la liste de balisage auto et peuvent être ajoutées pour le balisage automatique:
 ["Jean Baptiste de l'Aquila"]

Pour l'identifiant ArmandJeanDuPlessisDeRichelieu des personnes, les variantes d'écriture suivantes ne sont pas dans la liste de balisage auto et peuvent être ajoutées pour le balisage automatique:
 ['M. le Cardinal']

Pour l'identifiant IsabelleClaireEugenieDAutriche des personnes, les variantes d'écriture suivantes ne sont pas dans la liste de balisage auto et peuvent être ajoutées pour le balisage automatique:
 ['Princesse', "l'Infante", 'Infante']

Pour l'identifiant LouisIPhelypeauxDeLaVrilliere des personnes, les variantes d'écriture suivantes ne sont pas dans la liste de balisage auto et peuvent être ajoutées pour le balisage automatique:
 ['Monsieur de']

Pour l'identifiant PierreLouisReichDePennautier des personnes, les variantes d'écriture suiv

In [36]:
# Vérifier et améliorer les listes auteurs.csv et lieux.csv pour le balisage automatique 
# récupérer la liste des identifiants et des variantes dans auteurs.csv et lieux.csv
with open(pers_balise_auto, 'r', encoding='utf-8') as sheet:
    reader = csv.DictReader(sheet)
    pers_balise_auto_dict = {row['ID'].strip(): row['Noms'].strip() for row in reader if row['ID'] and row['Noms']}
with open(place_balise_auto, 'r', encoding='utf-8') as sheet:
    reader = csv.DictReader(sheet)
    place_balise_auto_dict = {row['ID'].strip(): row['Noms'].strip() for row in reader if row['ID'] and row['Noms']}

# comparer avec une liste des variantes trouvées dans les textes qui sont pas dans les listes d'ambiguites. C'est à dire des variantes qui ne sont pas présentes pour plus d'un identifiants, et pas d'ambiguité entre pers et place
var_pers_non_ambigue = {ref: noms for ref, noms in var_pers_dict.items() if all(nom not in ambiguites for nom in noms)}
var_place_non_ambigue = {ref: noms for ref, noms in var_place_dict.items() if all(nom not in ambiguites for nom in noms)}

print("\nVariantes d'écriture pour les personnes sans ambiguité:")
for ref, noms in var_pers_non_ambigue.items():
    print(f"  {ref}: {noms}")

print("\nVariantes d'écriture pour les lieux sans ambiguité:")
for ref, noms in var_place_non_ambigue.items():
    print(f"  {ref}: {noms}")



Variantes d'écriture pour les personnes sans ambiguité:
  ManiusValeriusMaximusCorvinusMessalla: ['Valerius Messala', 'Mexala']
  GiovanniBattistaBranconioDellAquila: ["Jean Baptiste de l'Aquila"]
  LodovicoDiLeonardoBuonarrotiSimoni: ['Loüis Bonarotti Simoni']
  LuciusManliusCapitolinusImperiosus: ['Lucio Manilio']
  QuintusFabiusMaximusVerrucosus: ['Fabius Maximus', 'F.Maximus', 'Fabius']
  ArmandJeanDuPlessisDeRichelieu: ['Cardinal Duc de Richelieu', 'Cardinale di Richelieu', 'Cardinal de Richelieu', 'M. le Cardinal']
  IsabelleClaireEugenieDAutriche: ["l'lnfante Isabelle", 'Infante Isabelle', 'Princesse', "l'Infante", 'Infante']
  AnthonieVanMontfoortBlocklandt: ['Antoine de Montfort de Blocland']
  GiovanniPaoloGallucciSalodiano: ['M. Giovanni Paolo Gallucci Salodiano']
  QuintusFabiusMaximusRullianus: ['Quintus Fabius Rutilien']
  GastonDeSecondatDeMontesquieu: ['Baron de Montesquieu']
  LouisIPhelypeauxDeLaVrilliere: ['Monsieur de']
  GiovanniAndreaGiliodaFabriano: ['Messer Gio

# vérif identifiants de paragraphes

In [8]:
from lxml import etree
from pathlib import Path
from collections import defaultdict

DIR = Path('../corpus-ArTerm/Architecture')

def check_ids(xml_path):
    tree = etree.parse(str(xml_path))
    
    # Récupère tous les attributs xml:id du document
    ids = [el.get('{http://www.w3.org/XML/1998/namespace}id') 
           for el in tree.iter() 
           if el.get('{http://www.w3.org/XML/1998/namespace}id') is not None]
    
    # Détecte les doublons
    counts = defaultdict(int)
    for id_ in ids:
        counts[id_] += 1
    
    duplicates = {id_: n for id_, n in counts.items() if n > 1}
    return ids, duplicates

# --- Rapport global ---
all_ids = defaultdict(list)  # id → liste des fichiers qui le contiennent
total_duplicates = 0

for xml_file in sorted(DIR.glob('*.xml')):
    ids, duplicates = check_ids(xml_file)
    
    status = f"  {len(ids)} id{'s' if len(ids) > 1 else ''} trouvé{'s' if len(ids) > 1 else ''}"
    if duplicates:
        total_duplicates += len(duplicates)
        status += f"  ⚠️  {len(duplicates)} doublon(s) :"
        for id_, n in duplicates.items():
            status += f"\n      → '{id_}' apparaît {n} fois"
    else:
        status += "  ✓ aucun doublon"
    
    print(f"\n{xml_file.name}")
    print(status)

    # Indexe pour la vérification inter-fichiers
    for id_ in ids:
        all_ids[id_].append(xml_file.name)

# --- Vérification inter-fichiers ---
cross_duplicates = {id_: files for id_, files in all_ids.items() if len(files) > 1}

print("\n" + "="*50)
if cross_duplicates:
    print(f"⚠️  {len(cross_duplicates)} id(s) partagé(s) entre plusieurs fichiers :")
    for id_, files in cross_duplicates.items():
        print(f"  → '{id_}' dans : {', '.join(files)}")
else:
    print("✓ Aucun doublon inter-fichiers")

print(f"\nBilan : {total_duplicates} doublon(s) intra-fichier{'s' if total_duplicates > 1 else ''} détecté(s) sur {len(list(DIR.glob('*.xml')))} fichiers.")


Barbaro_DieciLibriDellArchitetturaATIR.xml
  997 ids trouvés  ✓ aucun doublon

Bartoli_ArchitetturaATIR.xml
  289 ids trouvés  ✓ aucun doublon

Bassi_DispareriArchitettura.xml
  59 ids trouvés  ✓ aucun doublon

Cataneo_PrimiQuatroLibriArchitetturaATIR.xml
  164 ids trouvés  ✓ aucun doublon

Felibien_PrincipesArchitectureSculpturePeinture.xml
  4366 ids trouvés  ✓ aucun doublon

Huret_RegleDescrireColomnes.xml
  138 ids trouvés  ✓ aucun doublon

Martin_ArchitectureAlberti.xml
  2297 ids trouvés  ✓ aucun doublon

Martin_ArchitectureSerlio.xml
  280 ids trouvés  ✓ aucun doublon

Martin_DiscoursSongePoliphile.xml
  527 ids trouvés  ✓ aucun doublon

Palladio_ArchitectureTradFreart.xml
  445 ids trouvés  ✓ aucun doublon

Palladio_DescritioneChieseATIR.xml
  159 ids trouvés  ✓ aucun doublon

Palladio_QuattroLibriArchitetturaATIR.xml
  357 ids trouvés  ✓ aucun doublon

Raphael_LetteraLeoneX.xml
  22 ids trouvés  ✓ aucun doublon

Sambin_DiversiteTermesArchitecture.xml
  44 ids trouvés  ✓ aucun